In [ ]:
# ── Task 1: Create & Justify Engineered Features ──

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# ── Load data (same as Day 1) ──
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age','workclass','fnlwgt','education','education_num','marital_status',
           'occupation','relationship','race','sex','capital_gain','capital_loss',
           'hours_per_week','native_country','income']
df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)

# ── Same train/test split as Day 1 (stratified, random_state=42) ──
X = df.drop('income', axis=1)
y = (df['income'] == '>50K').astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Positive rate: {y_train.mean():.4f} (train), {y_test.mean():.4f} (test)')

# ── Create 7 Engineered Features ──

def engineer_features(df):
    """Create engineered features from raw columns. Uses only current-row data (no leakage)."""
    out = pd.DataFrame(index=df.index)

    # 1. has_capital_gain: binary flag
    #    Most people have capital_gain=0. The gap between 0 and any gain is what matters.
    out['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)

    # 2. log_capital_gain: log(1 + capital_gain)
    #    Capital gain is wildly skewed (0 to 99999). Log compresses the range.
    out['log_capital_gain'] = np.log1p(df['capital_gain'])

    # 3. has_capital_loss: binary flag
    #    Same logic as gain — presence/absence matters more than exact value.
    out['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)

    # 4. age_group: life-cycle bins
    #    Income follows a life cycle (low young → peaks mid-career → drops after retirement).
    out['age_group'] = pd.cut(
        df['age'],
        bins=[0, 24, 39, 54, 64, 100],
        labels=['<25', '25-39', '40-54', '55-64', '65+']
    )

    # 5. hours_category: work intensity bins
    #    Part-time vs full-time vs overtime → different income brackets.
    out['hours_category'] = pd.cut(
        df['hours_per_week'],
        bins=[0, 19, 39, 49, 100],
        labels=['<20', '20-39', '40-49', '50+']
    )

    # 6. higher_ed: binary flag (Bachelors or above)
    #    Clear threshold: college degree vs no degree.
    out['higher_ed'] = (df['education_num'] >= 14).astype(int)

    # 7. edu_hours_interaction: education_num × hours_per_week
    #    Combination: highly educated + long hours = highest earners.
    out['edu_hours_interaction'] = df['education_num'] * df['hours_per_week']

    return out

# ── Apply to train and test ──
eng_train = engineer_features(X_train)
eng_test = engineer_features(X_test)

print('\nEngineered features created:')
print(eng_train.columns.tolist())
print(f'Shape: {eng_train.shape}')
display(eng_train.head())

# ── Compute Mutual Information scores ──
# MI measures: how much does knowing this feature reduce uncertainty about income?
# MI = 0 → feature is useless. Higher = more informative.
# Need numeric-only input for MI (age_group and hours_category are categorical)
eng_train_numeric = pd.get_dummies(eng_train, drop_first=True)

mi_scores = mutual_info_classif(eng_train_numeric, y_train, random_state=42)

# Build feature dictionary
feature_dict = pd.DataFrame({
    'Feature': eng_train_numeric.columns,
    'MI_Score': np.round(mi_scores, 4)
}).sort_values('MI_Score', ascending=False).reset_index(drop=True)

print('\nFeature Dictionary — Mutual Information Scores:')
print('(Higher MI = more informative about income >50K)\n')
display(feature_dict)
print(f'\nBest feature: {feature_dict.iloc[0]["Feature"]} (MI={feature_dict.iloc[0]["MI_Score"]})')